# 09 — Feature Engineering & Imbalanced Data

Selecting the most useful features and handling datasets where classes are not evenly represented.

## Feature Selection: Filter Methods

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, chi2, mutual_info_classif

selector = SelectKBest(score_func=f_classif, k=10)   # ANOVA F-value, for classification
X_selected = selector.fit_transform(X_train, y_train)
selector.get_support(indices=True)   # indices of selected features
selector.scores_

# other score functions
SelectKBest(chi2, k=10)                  # for non-negative features (e.g. counts)
SelectKBest(mutual_info_classif, k=10)   # captures nonlinear relationships


## Feature Selection: Wrapper Methods (RFE)

In [ ]:
from sklearn.feature_selection import RFE, RFECV
from sklearn.linear_model import LogisticRegression

rfe = RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=10, step=1)
rfe.fit(X_train, y_train)
rfe.support_        # boolean mask of selected features
rfe.ranking_        # 1 = selected, higher = eliminated earlier

# automatically find optimal number of features via CV
rfecv = RFECV(estimator=LogisticRegression(max_iter=1000), cv=5, scoring="accuracy")
rfecv.fit(X_train, y_train)
rfecv.n_features_


## Feature Selection: Embedded Methods (model-based)

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

sfm = SelectFromModel(RandomForestClassifier(n_estimators=200), threshold="median")
sfm.fit(X_train, y_train)
X_selected = sfm.transform(X_train)

# Lasso naturally performs feature selection via L1 sparsity
from sklearn.linear_model import LassoCV
lasso_selector = SelectFromModel(LassoCV(cv=5))
lasso_selector.fit(X_train, y_train)


## Variance Threshold (remove near-constant features)

In [ ]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.01)
X_filtered = vt.fit_transform(X_train)


## Class Weights (handle imbalance without resampling)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

clf = LogisticRegression(class_weight="balanced")   # auto: weight inversely proportional to class frequency
clf.fit(X_train, y_train)

# manual weights
weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), weights))
clf_manual = LogisticRegression(class_weight=class_weight_dict)

# tree ensembles support it too
from sklearn.ensemble import RandomForestClassifier
RandomForestClassifier(class_weight="balanced_subsample")


## Resampling with imbalanced-learn

In [ ]:
# pip install imbalanced-learn
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)   # synthetic minority samples

under = RandomUnderSampler(random_state=42)
X_under, y_under = under.fit_resample(X_train, y_train)

# resampling must happen INSIDE cross-validation to avoid leakage — use imblearn's Pipeline
imb_pipe = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("clf", RandomForestClassifier(random_state=42)),
])
imb_pipe.fit(X_train, y_train)


## Combining Over- and Under-sampling

In [ ]:
from imblearn.combine import SMOTEENN, SMOTETomek

smote_enn = SMOTEENN(random_state=42)
X_res, y_res = smote_enn.fit_resample(X_train, y_train)


## Evaluating Imbalanced Models Correctly

In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score
from imblearn.metrics import classification_report_imbalanced

balanced_accuracy_score(y_test, y_pred)   # average recall across classes
f1_score(y_test, y_pred, average="macro")

print(classification_report_imbalanced(y_test, y_pred))   # includes geometric mean, specificity


## Common Gotchas
- Never apply SMOTE/oversampling before splitting into train/test — it leaks synthetic near-duplicates of test samples into training.
- `class_weight="balanced"` is often a simpler, equally effective first try compared to resampling.
- High-cardinality categorical features can dominate `f_classif`/`chi2` selection scores — check for that before trusting rankings.
- `RFE`/`RFECV` refit the estimator many times — expensive on large datasets or slow models; prefer `SelectFromModel` for a faster embedded alternative.